
# StreamVault Licensing Analysis

## Business objective

Support StreamVault's $14 million film licensing decision by comparing franchise collections with standalone movie candidates.

## Questions to answer

1. How do franchise titles compare with standalone proxies across all years?
2. How does the comparison change for titles released from 2000 onward?
3. Which three franchise collections and five standalone titles merit negotiation review?
4. Which directors and lead actors provide useful historical performance signals?
5. What should StreamVault consider for its strongest franchise candidate?

## Analytical approach

- Use the processed TMDb movie and credits data.
- Preserve one row per movie in the main analysis.
- Compare both required time windows.
- Document assumptions, exclusions, and limitations.
- Export reproducible evidence for the client memo.

## Limitations

Historical theatrical revenue is not streaming revenue, licensing cost, or profit. Films without a recorded collection are treated as standalone proxies, not confirmed standalone licensing opportunities.


In [1]:

from pathlib import Path

import pandas as pd
from time import perf_counter

ANALYSIS_STARTED = perf_counter()

# Locate the project root from the notebook's working directory.
PROJECT_ROOT = (
    Path.cwd()
    if (Path.cwd() / "id_list.csv").exists()
    else Path.cwd().parent
)

DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUT_DIR = PROJECT_ROOT / "out"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load the processed datasets.
movies = pd.read_csv(DATA_DIR / "movies.csv")
genres = pd.read_csv(DATA_DIR / "movie_genres.csv")
cast = pd.read_csv(DATA_DIR / "cast.csv")
directors = pd.read_csv(DATA_DIR / "directors.csv")

print("Processed data directory:", "data/processed/")
print("Movies:", movies.shape)
print("Genres:", genres.shape)
print("Cast:", cast.shape)
print("Directors:", directors.shape)


Processed data directory: data/processed/
Movies: (112, 11)
Genres: (320, 3)
Cast: (7827, 5)
Directors: (122, 3)


In [2]:

# Check primary-key uniqueness in the main movie table.
print("Movie rows:", len(movies))
print("Unique movie IDs:", movies["movie_id"].nunique())
print("Duplicate movie IDs:", movies["movie_id"].duplicated().sum())

# Check whether supporting records refer to known movies.
movie_ids = set(movies["movie_id"])

for name, frame in {
    "genres": genres,
    "cast": cast,
    "directors": directors,
}.items():
    unknown_ids = set(frame["movie_id"].dropna()) - movie_ids
    print(f"{name} records linked to unknown movies:", len(unknown_ids))

# Check the fields needed for the comparison.
print("\nLicensing categories:")
print(movies["licensing_type"].value_counts(dropna=False))

print("\nMissing release years:", movies["release_year"].isna().sum())
print("Missing revenue values:", movies["revenue"].isna().sum())
print("Missing collection names:", movies["collection_name"].isna().sum())


Movie rows: 112
Unique movie IDs: 112
Duplicate movie IDs: 0
genres records linked to unknown movies: 0
cast records linked to unknown movies: 0
directors records linked to unknown movies: 0

Licensing categories:
licensing_type
franchise           96
standalone_proxy    16
Name: count, dtype: int64

Missing release years: 0
Missing revenue values: 0
Missing collection names: 16


In [3]:

# Compare franchise movies with standalone proxies.

comparison_data = movies.copy()

comparison_data["release_year"] = pd.to_numeric(
    comparison_data["release_year"], errors="coerce"
)
comparison_data["revenue"] = pd.to_numeric(
    comparison_data["revenue"], errors="coerce"
)
comparison_data["vote_average"] = pd.to_numeric(
    comparison_data["vote_average"], errors="coerce"
)
comparison_data["vote_count"] = pd.to_numeric(
    comparison_data["vote_count"], errors="coerce"
)


def compare_licensing_types(data, period):
    summary = (
        data.groupby("licensing_type", dropna=False)
        .agg(
            movie_count=("movie_id", "nunique"),
            median_revenue=("revenue", "median"),
            mean_revenue=("revenue", "mean"),
            median_rating=("vote_average", "median"),
            median_vote_count=("vote_count", "median"),
        )
        .reset_index()
    )

    summary.insert(0, "period", period)
    return summary


all_years = compare_licensing_types(
    comparison_data,
    "All years",
)

from_2000 = compare_licensing_types(
    comparison_data.loc[
        comparison_data["release_year"] >= 2000
    ],
    "2000 onward",
)

comparison_results = pd.concat(
    [all_years, from_2000],
    ignore_index=True,
)

comparison_results


,period,licensing_type,movie_count,median_revenue,mean_revenue,median_rating,median_vote_count
0,All years,franchise,96,578937111.0,6.545567e+08,7.601,15482.5
1,All years,standalone_proxy,16,161802536.5,2.702712e+08,8.444,18854.5
2,2000 onward,franchise,63,786636033.0,7.929864e+08,7.450,15995.0
3,2000 onward,standalone_proxy,8,283195047.5,3.538082e+08,8.206,18854.5


In [4]:
# Rank franchise collections for further evaluation.

franchise_movies = comparison_data.loc[
    comparison_data["licensing_type"] == "franchise"
].copy()

collection_summary = (
    franchise_movies
    .groupby(["collection_id", "collection_name"], dropna=False)
    .agg(
        movie_count=("movie_id", "nunique"),
        first_release_year=("release_year", "min"),
        last_release_year=("release_year", "max"),
        total_theatrical_revenue=("revenue", "sum"),
        median_movie_revenue=("revenue", "median"),
        median_rating=("vote_average", "median"),
        median_vote_count=("vote_count", "median"),
    )
    .reset_index()
    .sort_values(
        "total_theatrical_revenue",
        ascending=False,
    )
)

collection_summary.head(15)

,collection_id,collection_name,movie_count,first_release_year,last_release_year,total_theatrical_revenue,median_movie_revenue,median_rating,median_vote_count
23,86311.0,The Avengers Collection,4,2012,2019,7919284026,1.785615e+09,8.1660,31401.0
10,1241.0,Harry Potter Collection,8,2001,2011,7759777709,9.360860e+08,7.7680,22850.5
6,295.0,Pirates of the Caribbean Collection,5,2003,2017,4525525744,9.616912e+08,7.2810,16072.0
0,10.0,Star Wars Collection,6,1977,2005,4432411744,7.123982e+08,7.6895,17000.5
21,10194.0,Toy Story Collection,4,1995,2019,3039690868,7.823458e+08,7.7205,16025.0
12,2150.0,Shrek Collection,4,2001,2010,2983130036,7.804555e+08,6.8745,12060.0
26,121938.0,The Hobbit Collection,3,2012,2014,2935523356,9.584000e+08,7.3890,15995.0
2,119.0,The Lord of the Rings Collection,3,2001,2003,2916544743,9.262874e+08,8.4510,27846.0
24,87359.0,Mission: Impossible Collection,5,1996,2015,2779994012,5.463881e+08,7.0360,10106.0
4,263.0,The Dark Knight Collection,3,2005,2012,2463740489,1.008481e+09,7.8010,25331.0


In [5]:
# Check whether franchise rankings change from 2000 onward.

modern_franchise_movies = comparison_data.loc[
    (comparison_data["licensing_type"] == "franchise")
    & (comparison_data["release_year"] >= 2000)
].copy()

modern_collection_summary = (
    modern_franchise_movies
    .groupby(["collection_id", "collection_name"], dropna=False)
    .agg(
        movie_count=("movie_id", "nunique"),
        first_release_year=("release_year", "min"),
        last_release_year=("release_year", "max"),
        total_theatrical_revenue=("revenue", "sum"),
        median_movie_revenue=("revenue", "median"),
        median_rating=("vote_average", "median"),
        median_vote_count=("vote_count", "median"),
    )
    .reset_index()
    .sort_values(
        "total_theatrical_revenue",
        ascending=False,
    )
    .reset_index(drop=True)
)

modern_collection_summary.head(15)

,collection_id,collection_name,movie_count,first_release_year,last_release_year,total_theatrical_revenue,median_movie_revenue,median_rating,median_vote_count
0,86311.0,The Avengers Collection,4,2012,2019,7919284026,1.785615e+09,8.1660,31401.0
1,1241.0,Harry Potter Collection,8,2001,2011,7759777709,9.360860e+08,7.7680,22850.5
2,295.0,Pirates of the Caribbean Collection,5,2003,2017,4525525744,9.616912e+08,7.2810,16072.0
3,2150.0,Shrek Collection,4,2001,2010,2983130036,7.804555e+08,6.8745,12060.0
4,121938.0,The Hobbit Collection,3,2012,2014,2935523356,9.584000e+08,7.3890,15995.0
5,119.0,The Lord of the Rings Collection,3,2001,2003,2916544743,9.262874e+08,8.4510,27846.0
6,263.0,The Dark Knight Collection,3,2005,2012,2463740489,1.008481e+09,7.8010,25331.0
7,131292.0,Iron Man Collection,3,2008,2013,2424684758,6.239333e+08,6.9320,24172.0
8,87359.0,Mission: Impossible Collection,4,2000,2015,2322297621,6.145524e+08,6.9445,9053.0
9,10194.0,Toy Story Collection,2,2010,2019,2141157495,1.070579e+09,7.6385,13808.0


In [6]:
# Compare standalone proxies across both required time windows.

standalone_movies = comparison_data.loc[
    comparison_data["licensing_type"] == "standalone_proxy"
].copy()

candidate_columns = [
    "movie_id",
    "title",
    "release_year",
    "revenue",
    "vote_average",
    "vote_count",
]

print("TOP STANDALONE PROXIES — ALL YEARS")

display(
    standalone_movies[candidate_columns]
    .sort_values(
        ["revenue", "vote_count"],
        ascending=False,
    )
    .head(10)
)

print("TOP STANDALONE PROXIES — 2000 ONWARD")

display(
    standalone_movies.loc[
        standalone_movies["release_year"] >= 2000,
        candidate_columns,
    ]
    .sort_values(
        ["revenue", "vote_count"],
        ascending=False,
    )
    .head(10)
)

TOP STANDALONE PROXIES — ALL YEARS


,movie_id,title,release_year,revenue,vote_average,vote_count
37,27205,Inception,2010,839030630,8.400,40754
108,157336,Interstellar,2014,746606706,8.493,41925
66,13,Forrest Gump,1994,677387716,8.462,30907
54,10681,WALL·E,2008,521311860,8.129,21202
96,424,Schindler's List,1993,321365567,8.600,18195
36,1422,The Departed,2006,291465000,8.160,16939
63,129,Spirited Away,2001,274925095,8.537,19231
6,680,Pulp Fiction,1994,213928762,8.480,31365
50,1124,The Prestige,2006,109676311,8.212,18478
71,550,Fight Club,1999,100853753,8.437,33383


TOP STANDALONE PROXIES — 2000 ONWARD


,movie_id,title,release_year,revenue,vote_average,vote_count
37,27205,Inception,2010,839030630,8.400,40754
108,157336,Interstellar,2014,746606706,8.493,41925
54,10681,WALL·E,2008,521311860,8.129,21202
36,1422,The Departed,2006,291465000,8.160,16939
63,129,Spirited Away,2001,274925095,8.537,19231
50,1124,The Prestige,2006,109676311,8.212,18478
75,77,Memento,2000,40060108,8.200,17054
55,641,Requiem for a Dream,2000,7390108,8.007,11494


In [7]:
# Compare the three leading franchise candidates.

priority_collections = [
    "The Avengers Collection",
    "Harry Potter Collection",
    "Pirates of the Caribbean Collection",
]

collection_comparison = collection_summary.loc[
    collection_summary["collection_name"].isin(priority_collections)
].copy()

collection_comparison[
    [
        "collection_name",
        "movie_count",
        "first_release_year",
        "last_release_year",
        "total_theatrical_revenue",
        "median_movie_revenue",
        "median_rating",
        "median_vote_count",
    ]
].sort_values(
    "median_movie_revenue",
    ascending=False,
)

,collection_name,movie_count,first_release_year,last_release_year,total_theatrical_revenue,median_movie_revenue,median_rating,median_vote_count
23,The Avengers Collection,4,2012,2019,7919284026,1.785615e+09,8.166,31401.0
6,Pirates of the Caribbean Collection,5,2003,2017,4525525744,9.616912e+08,7.281,16072.0
10,Harry Potter Collection,8,2001,2011,7759777709,9.360860e+08,7.768,22850.5


In [8]:
# Compare the top five standalone proxies.

period = "all_years"  # Change to "since_2000" for the 2000-onward analysis

standalone_movies = comparison_data.loc[
    comparison_data["licensing_type"] == "standalone_proxy"
].copy()

if period == "since_2000":
    standalone_movies = standalone_movies.loc[
        standalone_movies["release_year"] >= 2000
    ]

standalone_top_5 = (
    standalone_movies.sort_values(
        ["revenue", "vote_average", "vote_count"],
        ascending=[False, False, False],
    )
    .head(5)
    .copy()
)

standalone_top_5.insert(0, "rank", range(1, len(standalone_top_5) + 1))

standalone_top_5[
    [
        "rank",
        "movie_id",
        "title",
        "release_year",
        "revenue",
        "vote_average",
        "vote_count",
    ]
]

,rank,movie_id,title,release_year,revenue,vote_average,vote_count
37,1,27205,Inception,2010,839030630,8.400,40754
108,2,157336,Interstellar,2014,746606706,8.493,41925
66,3,13,Forrest Gump,1994,677387716,8.462,30907
54,4,10681,WALL·E,2008,521311860,8.129,21202
96,5,424,Schindler's List,1993,321365567,8.600,18195


In [9]:
# Inspect the credits table structures before aggregating performance.
print("Director columns:", directors.columns.tolist())
print("Cast columns:", cast.columns.tolist())

print("\nDirector sample:")
display(directors.head())

print("\nCast sample:")
display(cast.head())

Director columns: ['movie_id', 'person_id', 'director_name']
Cast columns: ['movie_id', 'person_id', 'person_name', 'character', 'cast_order']

Director sample:


,movie_id,person_id,director_name
0,155,525,Christopher Nolan
1,165,24,Robert Zemeckis
2,242,1776,Francis Ford Coppola
3,49026,525,Christopher Nolan
4,8077,7467,David Fincher



Cast sample:


,movie_id,person_id,person_name,character,cast_order
0,155,3894,Christian Bale,Bruce Wayne,0
1,155,1810,Heath Ledger,Joker,1
2,155,6383,Aaron Eckhart,Harvey Dent,2
3,155,3895,Michael Caine,Alfred,3
4,155,1579,Maggie Gyllenhaal,Rachel,4


In [10]:
# Analyse director and lead-actor performance.
# Each movie contributes its revenue only once per person.

movie_metrics = comparison_data[
    [
        "movie_id",
        "title",
        "release_year",
        "revenue",
        "vote_average",
        "vote_count",
    ]
].drop_duplicates(subset="movie_id")


# 1. Director performance
director_movies = directors.merge(
    movie_metrics,
    on="movie_id",
    how="inner",
    validate="many_to_one",
)

director_summary = (
    director_movies
    .groupby("director_name")
    .agg(
        movie_count=("movie_id", "nunique"),
        median_revenue=("revenue", "median"),
        mean_revenue=("revenue", "mean"),
        median_rating=("vote_average", "median"),
        median_vote_count=("vote_count", "median"),
    )
    .reset_index()
    .query("movie_count >= 2")
    .sort_values(
        ["median_revenue", "movie_count"],
        ascending=[False, False],
    )
)

print("TOP DIRECTORS (at least 2 movies)")
display(director_summary.head(10))


# 2. Lead-actor performance
lead_cast = (
    cast.loc[cast["cast_order"] == 0]
    .drop_duplicates(subset="movie_id")
)

lead_actor_movies = lead_cast.merge(
    movie_metrics,
    on="movie_id",
    how="inner",
    validate="one_to_one",
)

lead_actor_summary = (
    lead_actor_movies
    .groupby("person_name")
    .agg(
        movie_count=("movie_id", "nunique"),
        median_revenue=("revenue", "median"),
        mean_revenue=("revenue", "mean"),
        median_rating=("vote_average", "median"),
        median_vote_count=("vote_count", "median"),
    )
    .reset_index()
    .query("movie_count >= 2")
    .sort_values(
        ["median_revenue", "movie_count"],
        ascending=[False, False],
    )
)

print("TOP LEAD ACTORS (at least 2 movies)")
display(lead_actor_summary.head(10))

TOP DIRECTORS (at least 2 movies)


,director_name,movie_count,median_revenue,mean_revenue,median_rating,median_vote_count
3,Anthony Russo,2,2.497532e+09,2.497532e+09,8.2470,31401.0
31,Joe Russo,2,2.497532e+09,2.497532e+09,8.2470,31401.0
39,Joss Whedon,2,1.462110e+09,1.462110e+09,7.6815,32877.5
22,Gore Verbinski,3,9.616912e+08,8.942941e+08,7.3930,17859.0
51,Peter Jackson,6,9.572099e+08,9.753447e+08,8.0060,22675.0
7,Chris Columbus,2,9.530315e+08,9.530315e+08,7.8075,27532.5
15,David Yates,4,9.462593e+08,1.041997e+09,7.7070,21478.0
20,George Lucas,4,8.126990e+08,8.303279e+08,7.0380,15975.5
6,Carlos Saldanha,2,7.738428e+08,7.738428e+08,6.7580,9933.0
11,Christopher Nolan,7,7.466067e+08,5.998735e+08,8.2120,25331.0


TOP LEAD ACTORS (at least 2 movies)


,person_name,movie_count,median_revenue,mean_revenue,median_rating,median_vote_count
37,Robert Downey Jr.,7,1.405404e+09,1.477710e+09,7.6680,29204.0
22,Johnny Depp,5,9.616912e+08,9.051051e+08,7.2810,16072.0
17,Ian McKellen,2,9.572099e+08,9.572099e+08,7.4600,15409.0
8,Daniel Radcliffe,8,9.360860e+08,9.699722e+08,7.7680,22850.5
10,Elijah Wood,3,9.262874e+08,9.721816e+08,8.4510,27846.0
34,Mike Myers,4,7.804555e+08,7.457825e+08,6.8745,12060.0
15,Hayden Christensen,2,7.496992e+08,7.496992e+08,7.0380,15207.0
40,Sam Neill,2,7.136175e+08,7.136175e+08,7.0780,13409.0
5,Christian Bale,4,6.913496e+08,7.087734e+08,7.7670,24437.5
26,Liam Neeson,2,6.839405e+08,6.839405e+08,7.5910,17315.0


In [11]:
# Export the evidence tables used in the client memo.
OUT_DIR.mkdir(parents=True, exist_ok=True)
comparison_results.to_csv(OUT_DIR / "comparison_results.csv", index=False)
collection_summary.head(10).to_csv(OUT_DIR / "franchise_rankings_all_years.csv", index=False)
modern_collection_summary.head(10).to_csv(OUT_DIR / "franchise_rankings_2000_onward.csv", index=False)

standalone_all = standalone_movies.sort_values(
    ["revenue", "vote_average", "vote_count"], ascending=False
).head(5).copy()
standalone_recent = standalone_movies.loc[
    standalone_movies["release_year"] >= 2000
].sort_values(["revenue", "vote_average", "vote_count"], ascending=False).head(5).copy()
standalone_all.to_csv(OUT_DIR / "standalone_rankings_all_years.csv", index=False)
standalone_recent.to_csv(OUT_DIR / "standalone_rankings_2000_onward.csv", index=False)
director_summary.to_csv(OUT_DIR / "director_summary.csv", index=False)
lead_actor_summary.to_csv(OUT_DIR / "lead_actor_summary.csv", index=False)

assert movies["movie_id"].is_unique, "Movie table contains duplicate IDs"
for name, frame in {"genres": genres, "cast": cast, "directors": directors}.items():
    assert set(frame["movie_id"].dropna()).issubset(set(movies["movie_id"])), f"{name} has unknown movie IDs"
assert len(comparison_results) == 4, "Expected two categories in each of two periods"
print("Exported analysis evidence to:", "out/")
analysis_seconds = perf_counter() - ANALYSIS_STARTED
print("All-years and 2000-onward windows exported; movie joins validated.")
print(f"Analysis elapsed seconds (including data load): {analysis_seconds:.3f}")


Exported analysis evidence to: out/
All-years and 2000-onward windows exported; movie joins validated.
Analysis elapsed seconds (including data load): 0.387


In [12]:
# Final integrity checks: fail the notebook if its analysis or exports are inconsistent.
expected_categories = {"franchise", "standalone_proxy"}
expected_periods = {"All years", "2000 onward"}

assert movies["movie_id"].notna().all(), "Movie table contains a missing movie ID"
assert movies["movie_id"].is_unique, "Movie table contains duplicate movie IDs"
assert set(movies["licensing_type"].dropna()) == expected_categories, "Unexpected licensing categories"
assert movies["licensing_type"].notna().all(), "Movie has no licensing category"
assert (movies["revenue"].dropna() >= 0).all(), "Movie table contains negative revenue"

movie_ids = set(movies["movie_id"])
for name, frame in {"genres": genres, "cast": cast, "directors": directors}.items():
    assert frame["movie_id"].notna().all(), f"{name} contains a missing movie ID"
    assert set(frame["movie_id"]).issubset(movie_ids), f"{name} refers to a movie absent from movies"

assert set(comparison_results["period"]) == expected_periods, "Comparison is missing a required time period"
assert set(comparison_results["licensing_type"]) == expected_categories, "Comparison is missing a licensing category"
assert len(comparison_results) == len(expected_periods) * len(expected_categories), "Comparison has missing or duplicate period/category rows"
assert not comparison_results.duplicated(["period", "licensing_type"]).any(), "Comparison repeats a period/category row"
for period, frame in [("All years", comparison_data), ("2000 onward", comparison_data.loc[comparison_data["release_year"] >= 2000])]:
    observed = comparison_results.loc[comparison_results["period"] == period].set_index("licensing_type")["movie_count"]
    expected = frame.groupby("licensing_type")["movie_id"].nunique()
    assert observed.to_dict() == expected.to_dict(), f"Movie counts do not reconcile for {period}"

export_dir = PROJECT_ROOT / "out"
franchise_all_export = pd.read_csv(export_dir / "franchise_rankings_all_years.csv")
franchise_recent_export = pd.read_csv(export_dir / "franchise_rankings_2000_onward.csv")
standalone_all_export = pd.read_csv(export_dir / "standalone_rankings_all_years.csv")
standalone_recent_export = pd.read_csv(export_dir / "standalone_rankings_2000_onward.csv")
expected_standalone_all = standalone_movies.sort_values(["revenue", "vote_average", "vote_count"], ascending=False).head(5)
expected_standalone_recent = standalone_movies.loc[standalone_movies["release_year"] >= 2000].sort_values(["revenue", "vote_average", "vote_count"], ascending=False).head(5)
assert franchise_all_export["collection_name"].tolist() == collection_summary.head(10)["collection_name"].tolist(), "All-years franchise export does not match notebook ranking"
assert franchise_recent_export["collection_name"].tolist() == modern_collection_summary.head(10)["collection_name"].tolist(), "Recent franchise export does not match notebook ranking"
assert standalone_all_export["movie_id"].tolist() == expected_standalone_all["movie_id"].tolist(), "All-years standalone export does not match notebook ranking"
assert standalone_recent_export["movie_id"].tolist() == expected_standalone_recent["movie_id"].tolist(), "Recent standalone export does not match notebook ranking"
assert standalone_recent_export["release_year"].ge(2000).all(), "Recent standalone export includes a pre-2000 title"

transform_status = pd.read_csv(DATA_DIR / "transform_status.csv", dtype={"movie_id": str})
extraction_status = pd.read_csv(DATA_DIR / "extraction_status.csv", dtype={"movie_id": str})
assert len(transform_status) == len(extraction_status), "Extraction and transformation audit row counts differ"
assert transform_status["source_row"].is_unique, "Transformation audit repeats a source row"
assert extraction_status["source_row"].is_unique, "Extraction audit repeats a source row"
included_status = transform_status.loc[transform_status["transform_status"] == "included"]
assert len(included_status) == len(movies), "Transformation included count does not match the movie table"
assert included_status["movie_id"].is_unique, "Transformation audit includes a movie ID more than once"
assert set(included_status["movie_id"].astype("int64")) == set(movies["movie_id"].astype("int64")), "Transformation included IDs do not match the movie table"

print("All final integrity checks passed.")
print(f"Verified {len(movies)} unique movies; {len(genres)} genre links, {len(cast)} cast links, and {len(directors)} director links.")
print("Verified both time windows, both licensing categories, all four ranking exports, and the transformation audit.")


All final integrity checks passed.
Verified 112 unique movies; 320 genre links, 7827 cast links, and 122 director links.
Verified both time windows, both licensing categories, all four ranking exports, and the transformation audit.
